# Practice 097 — Econometrics: OLS from Scratch & the Gauss-Markov Assumptions

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's folder, so the practice root --
# where the `src` package lives -- is not on sys.path. Put it there.
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import load_dataset
from src.plotting import coefficient_plot, residual_diagnostic_plot, sampling_distribution_plot

## Phase 1 — OLS from scratch via QR decomposition

We simulate data with a **known** true `beta` (see `src/datasets.py`), then estimate
it two ways: our own QR-based estimator, and `statsmodels`. If Phase 1 is correct,
the two should agree to numerical precision — that agreement *is* the check, not a
separate test suite.

In [ ]:
data = load_dataset("homoskedastic", n=200, seed=0)
print(f"True beta: {data.beta_true}")
data.as_frame().head()

### Exercise — `src/_01_ols_qr.py :: ols_via_qr`

Open `src/_01_ols_qr.py`, read the `TODO(human)` block above the function,
implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_ols_qr import compare_to_statsmodels, ols_via_qr

compare_to_statsmodels(data.X, data.y)
fit = ols_via_qr(data.X, data.y)
fig = residual_diagnostic_plot(fit.fitted, fit.resid, title="Phase 1 — homoskedastic scenario")
fig

## Phase 2 — The sampling distribution of beta-hat

`beta_hat` is a random variable. Under Gauss-Markov assumptions A1-A5, its variance
has a closed form: `sigma_hat^2 * (X'X)^-1`. We check that formula against the
*empirical* variance of `beta_hat` across many simulated resamples — they should
match closely under the homoskedastic scenario.

### Exercise — `src/_02_sampling_distribution.py :: ols_vcov_homoskedastic`

Open `src/_02_sampling_distribution.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_sampling_distribution import ols_vcov_homoskedastic, simulate_beta_hat_distribution

vcov = ols_vcov_homoskedastic(data.X, fit.resid, fit.XtX_inv)
analytical_se = np.sqrt(np.diag(vcov))
print(f"Analytical SE (one sample):      {analytical_se}")

betas = simulate_beta_hat_distribution(n=200, n_sims=500, seed=1)
empirical_se = betas.std(axis=0, ddof=1)
print(f"Empirical SE (Monte Carlo, 500x): {empirical_se}")

fig = sampling_distribution_plot(betas[:, 1], true_value=data.beta_true[1], coef_name="beta_1 (x1)")
fig

## Phase 3 — Diagnosing heteroskedasticity

Assumption A5 says the error variance is constant across observations. We generate
a *heteroskedastic* dataset (error scale grows with `|x1|`) alongside the
homoskedastic one, and run the Breusch-Pagan LM test on both — it should reject
homoskedasticity only for the heteroskedastic scenario.

In [ ]:
hetero_data = load_dataset("heteroskedastic", n=300, seed=0)
hetero_fit = ols_via_qr(hetero_data.X, hetero_data.y)
fig = residual_diagnostic_plot(hetero_fit.fitted, hetero_fit.resid, title="Phase 3 — heteroskedastic scenario")
fig

### Exercise — `src/_03_heteroskedasticity.py :: breusch_pagan_lm_test`

Open `src/_03_heteroskedasticity.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._03_heteroskedasticity import breusch_pagan_lm_test

for label, d, f in (("homoskedastic", data, fit), ("heteroskedastic", hetero_data, hetero_fit)):
    lm, pval = breusch_pagan_lm_test(d.X, f.resid)
    print(f"{label:16s} LM={lm:7.3f}  p={pval:.4f}")

## Phase 4 — Heteroskedasticity-robust standard errors (HC0-HC3)

OLS `beta_hat` is still unbiased under heteroskedasticity — only Phase 2's classical
standard errors are wrong. White's sandwich estimator (and its HC1-HC3 finite-sample
corrections) fixes the standard errors without touching the point estimate.

### Exercise — `src/_04_robust_se.py :: hc_vcov`

Open `src/_04_robust_se.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._04_robust_se import hc_vcov

hc_se = {}
for kind in ("HC0", "HC1", "HC2", "HC3"):
    hc_vc = hc_vcov(hetero_data.X, hetero_fit.resid, hetero_fit.XtX_inv, kind=kind)
    hc_se[kind] = np.sqrt(np.diag(hc_vc))
    print(f"{kind}: SE = {hc_se[kind]}")

## Phase 5 — Clustered standard errors

HC0-HC3 still assume independent observations. When errors share a within-group
shock (the `clustered` scenario), even HC3 understates the true standard errors —
the cluster-robust (CR1) estimator fixes that by summing scores within each cluster
before building the sandwich.

### Exercise — `src/_05_clustered_se.py :: cluster_robust_vcov`

Open `src/_05_clustered_se.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._05_clustered_se import cluster_robust_vcov

clustered_data = load_dataset("clustered", n=300, seed=0)
clustered_fit = ols_via_qr(clustered_data.X, clustered_data.y)

cluster_vc = cluster_robust_vcov(
    clustered_data.X, clustered_fit.resid, clustered_fit.XtX_inv, clustered_data.cluster_id
)
cluster_se = np.sqrt(np.diag(cluster_vc))
print(f"Cluster-robust SE: {cluster_se}")

naive_vc = ols_vcov_homoskedastic(clustered_data.X, clustered_fit.resid, clustered_fit.XtX_inv)
naive_se = np.sqrt(np.diag(naive_vc))
print(f"Naive classical SE (wrong here): {naive_se}")

## End-to-end: comparing every standard-error method on one coefficient plot

Same point estimate (`beta_hat` on the heteroskedastic dataset), three standard-error
methods side by side. Watch how the CI widens as the method accounts for more of the
assumption violation.

In [ ]:
classical_vc = ols_vcov_homoskedastic(hetero_data.X, hetero_fit.resid, hetero_fit.XtX_inv)
se_by_method = {
    "classical": np.sqrt(np.diag(classical_vc)),
    "HC3": hc_se["HC3"],
}
fig = coefficient_plot(["intercept", "x1", "x2"], hetero_fit.beta, se_by_method)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert np.allclose(fit.beta, data.beta_true, atol=0.5), "beta_hat should be close to the true beta"
assert hc_se["HC3"].shape == (3,)
assert cluster_se.shape == (3,)
# Under real within-cluster correlation, ignoring it (naive) understates SEs
# relative to the cluster-robust estimate for at least one coefficient.
assert np.any(cluster_se > naive_se), "cluster-robust SEs should exceed the naive ones somewhere"
print("OK")